## Imports & Initializations

In [1]:
import requests
import mwparserfromhell
import re
import time
import html
import json
import uuid
import numpy as np
import pandas as pd
import duckdb as ddb
from matplotlib import pyplot as plt
from dotenv import dotenv_values
from tqdm import tqdm
import hashlib
from collections import deque
import asyncio
import inspect
import os
from tqdm.asyncio import tqdm_asyncio

import semchunk
from google import genai

In [2]:
# Get Env vars
config = dotenv_values("../.env")

In [3]:
# Gemini client
client = genai.Client(api_key=config["GEMINI_API_KEY"])

In [4]:
# Semchunk chunker
chunker = semchunk.chunkerify(lambda text: len(text.split()), 500)

## Functions

### Miscellaneous functions

In [5]:
def hash_text(text: str):
    text_bytes = text.encode('utf-8')
    hash_object = hashlib.sha256(text_bytes)
    hash_hex = hash_object.hexdigest()
    
    return hash_hex


### Gemini Question generation Functions

In [6]:
def ask_gemini(
    contents: str,
    model: str = "gemini-2.5-flash",
    retries: int = 10,
    backoff_factor: float = 2.0,
) -> str:
    """
    Generate text using Gemini with retry and standard exception handling.
    """
    for attempt in range(1, retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=contents,
            )
            return response.text.strip()

        except (ConnectionError, TimeoutError) as e:
            print(f"Network issue on attempt {attempt}: {e}")
        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")

        # Retry if not the last attempt
        if attempt < retries:
            sleep_time = backoff_factor ** (attempt - 1)
            print(f"Retrying in {sleep_time:.1f} seconds...")
            time.sleep(sleep_time)
        else:
            print("All retries failed.")
            return f"Error: {e}"

    return "Failed to generate response after multiple attempts."

In [7]:
def generate_questions(
    context: str,
    model: str = "gemini-2.5-flash",
    retries: int = 3,
    backoff_factor: float = 2.0,
) -> dict:
    """
    Generate Q&A pairs from a given context using Gemini,
    returning a JSON object like:
      {"QAs": [{"Question": "...", "Answer": "..."}]}
    """
    system_prompt = (
        "You are an expert question generator. "
        "Given the following text, generate 5 question-answer pairs "
        "that test understanding of its key ideas. "
        "Questions must be yes/no questions, and answers must only be yes or no."
        "Respond ONLY in valid JSON with this structure:\n"
        '{"QAs": [{"Question": "<string>", "Answer": "<string>"}]}'
    )

    for attempt in range(1, retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=[system_prompt, context],
                config={"response_mime_type": "application/json"},
            )

            # If response is valid JSON, .parsed gives a Python dict
            # print(response.text)
            return json.loads(response.text)

        except (ConnectionError, TimeoutError) as e:
            print(f"Network issue on attempt {attempt}: {e}")
        except Exception as e:
            print(f"Error on attempt {attempt}: {e}")

        if attempt < retries:
            sleep_time = backoff_factor ** (attempt - 1)
            print(f"Retrying in {sleep_time:.1f} seconds...")
            time.sleep(sleep_time)
        else:
            print("All retries failed.")
            return {"QAs": []}

    return {"QAs": []}

### Semantic Chunking Functions

In [8]:
def get_chunks(content: str, threshold: int = 400) -> list[str]:
    chunks = chunker(content)
    valid_chunks = [chunk for chunk in chunks if len(chunk.split()) > threshold]
    
    return valid_chunks

# Gutenberg dataset generation

In [9]:
import requests, re, os, time
from slugify import slugify

OUT_DIR = "/hpc/home/bfa6/work/github/yapper/dataset"
TARGET = 50
os.makedirs(OUT_DIR, exist_ok=True)

def clean(t):
    t = t.replace("\r\n","\n")
    m = re.search(r"(?mi)\*\*\* *START OF (THIS|THE) PROJECT GUTENBERG EBOOK.*?\*\*\*", t)
    if m: t = t[m.end():]
    m2 = re.search(r"(?mi)^start of (the|this) project gutenberg", t)
    if m2: t = t[m2.end():]
    m3 = re.search(r"(?mi)^\s*contents\s*$", t[:4000])
    chap = re.compile(r"(?mi)^(chapter|chap\.|book|part)\b.*", re.M)
    if m3:
        mc = chap.search(t[m3.end():])
        if mc: t = t[m3.end()+mc.start():]
    m4 = chap.search(t[:10000])
    if m4: t = t[m4.start():]
    first = t.split("\n\n",1)
    if len(first)==2 and re.search(r"(?i)project gutenberg|license|copyright", first[0]):
        t = first[1]
    return t.strip()

def pick(fmt):
    for k,v in fmt.items():
        if "text/plain" in k.lower(): return v
    return None

def download_books(n=TARGET):
    s = requests.Session()
    got = 0
    seen = set()
    saved = []
    url = "https://gutendex.com/books?languages=en&mime_type=text%2Fplain&sort=descending"
    while url and got < n:
        try:
            j = s.get(url, timeout=20).json()
        except Exception as e:
            print("page error:", e); break
        page = sorted(j.get("results",[]), key=lambda b: (b.get("download_count") or 0))
        for b in page:
            if got >= n: break
            bid = b.get("id")
            if not bid or bid in seen: 
                continue
            u = pick(b.get("formats",{}))
            if not u: 
                seen.add(bid); continue
            try:
                r = s.get(u, timeout=20)
                r.encoding = r.apparent_encoding or "utf-8"
                txt = clean(r.text)
                fn = os.path.join(OUT_DIR, slugify(f"{bid}-{b.get('title','untitled')}")[:150]+".txt")
                # avoid overwriting if file exists (very unlikely because of slug with id)
                if not os.path.exists(fn):
                    with open(fn,"w",encoding="utf8") as f: f.write(txt)
                    print(f"[{got+1}] {b.get('title')} (downloads: {b.get('download_count')})")
                    saved.append(fn); got += 1
                else:
                    print("exists, skip:", fn)
                seen.add(bid)
                time.sleep(0.25)
            except Exception as e:
                print("skip", bid, e)
                seen.add(bid)
        url = j.get("next")
        time.sleep(0.25)
    print(f"Done: saved {got} books to {OUT_DIR}")
    return saved


In [ ]:
files = download_books(200)
files[:5] 

[1] School education, Home Education Series, vol. 3 (of 6) (downloads: 0)
[2] The passing of the phantoms : $b A study of evolutionary psychology and morals (downloads: 0)
[3] Cats and kittens (downloads: 0)
[4] Anthropology and modern life (downloads: 0)
[5] The girl at Silver Thistle (downloads: 0)
[6] Bear and forbear : $b or, The young skipper of lake Ucayga (downloads: 0)
[7] The life of Abdel Kader, ex-sultan of the Arabs of Algeria (downloads: 0)
[8] The crusades (downloads: 0)
[9] Codes (downloads: 0)
[10] A diary of the wreck of His Majesty's ship Challenger, on the western coast of South America, in May, 1835 : $b with an account of the subsequent encampment of the officers and crew, during a period of seven weeks, on the south coast of Chili (downloads: 0)
[11] The adventures of Harlequin (downloads: 0)
[12] Walks and talks of an American farmer in England (Part 2 of 2) : $b In the years 1850-51. (downloads: 0)
[13] Walks and talks of an American farmer in England (Part 1 of

['/hpc/home/bfa6/work/github/yapper/dataset/77188-school-education-home-education-series-vol-3-of-6.txt',
 '/hpc/home/bfa6/work/github/yapper/dataset/77186-the-passing-of-the-phantoms-b-a-study-of-evolutionary-psychology-and-morals.txt',
 '/hpc/home/bfa6/work/github/yapper/dataset/77183-cats-and-kittens.txt',
 '/hpc/home/bfa6/work/github/yapper/dataset/77181-anthropology-and-modern-life.txt',
 '/hpc/home/bfa6/work/github/yapper/dataset/77180-the-girl-at-silver-thistle.txt']

In [ ]:
base = "/hpc/home/bfa6/work/github/yapper/dataset/books"
files = [base + "/" + file for file in os.listdir(base)]

In [21]:
all_chunks = []

for file in files:
    with open(file, "r") as f:
        book = f.read() 
    chunks = get_chunks(book)
    all_chunks += [{"chunk": chunk, "source": file} for chunk in chunks]

In [22]:
print(len(all_chunks))

18914


In [24]:
with open(base + "/chunks.json", "w") as f:
    json.dump(all_chunks, f) 

In [23]:
all_chunks[:1]

[{'chunk': '                         THE CAVE OF ELEPHANTA.\n\n [Illustration: A view of a cave, with large statues and pillars and two\n                         people standing inside.]\n\nOne of the earliest monuments of India that attracted the notice of\nEuropeans was the excavation of Elephanta, situated in a beautiful\nisland of the same name, called by the natives Goripura, or _Mountain\nCity_. This island is in the bay of Bombay, seven miles from Bombay\ncastle; it is about six miles in circumference, and composed of two long\nhills with a narrow valley between them.\n\nThe island has taken its familiar name from a colossal statue of an\nelephant, cut out of a detached mass of blackish rock unconnected with\nany stratum below. This figure has had another on its back, which the\nold travellers call a young elephant, but which, as far as we can judge\nfrom the drawing of what remains of it, has much more probably been a\ntiger. The head and neck of this elephant dropped off about

# Remove chunks with more than > 1024 tokens

In [10]:
# Imports

import os
import re

os.environ["TRANSFORMERS_CACHE"] = "/hpc/home/bfa6/work/llms/.cache"
os.environ["HF_HOME"] = "/hpc/home/bfa6/work/llms/.cache"

import time
import json

from unsloth import FastLanguageModel
import torch
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from dotenv import dotenv_values
from tqdm import tqdm
import asyncio
from tqdm.asyncio import tqdm_asyncio
from google import genai
import random
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
import vllm
import nltk
from matplotlib import pyplot as plt

config = dotenv_values("../.env")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/hpc/home/bfa6/work/github/yapper/.venv/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


INFO 11-26 10:20:56 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "unsloth/Qwen3-0.6B",
    cache_dir="/hpc/home/bfa6/work/llms/.cache",
)

In [ ]:
system_prompt = (
    "Your goal is to compress the information from the user in as few tokens as necessary and output the compressed version.\n"
    "Do NOT produce internal chain-of-thought or step-by-step reasoning.\n"
    "Start immediately with the compressed content (no extra preface)."
)
system_prompt

'You are given some context.\nYour goal is to compress the information in the context and output the compressed version.\nUse as few tokens as possible while keeping all information.\nDo NOT produce internal chain-of-thought or step-by-step reasoning.\nStart immediately with the compressed content (no extra preface).'

In [13]:
# load dataset
with open("/hpc/home/bfa6/work/github/yapper/dataset/chunks.json", "r") as f:
    dataset =  json.load(f)


len(dataset)

18914

In [28]:
def clean_whitespace(text: str) -> str:
    # Collapse multiple spaces/tabs into a single space
    text = re.sub(r'[ \t]+', ' ', text)
    # Collapse multiple newlines into a single newline
    text = re.sub(r'\n+', '\n', text)
    # Strip leading/trailing whitespace
    return text.strip()


for data in tqdm(dataset):
    data["chunk"] = clean_whitespace(data["chunk"])

100%|██████████| 18914/18914 [00:03<00:00, 5149.67it/s]


In [29]:
def get_text_len(text: str):
    inp = [{"role":"system", "content": system_prompt}, {"role":"user", "content": text}]
    text = tokenizer.apply_chat_template(
        inp,
        tokenize = False,
        add_generation_prompt=True,
        enable_thinking=False     
    )

    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    return len(inputs["input_ids"][0])

In [32]:
# Check token lengths
lens = []

for data in tqdm(dataset):
    length = get_text_len(data["chunk"])

    lens.append({"chunk": data["chunk"], "length": length})


df = pd.DataFrame.from_dict(lens)

df[df.length > 1024].chunk

100%|██████████| 18914/18914 [00:44<00:00, 423.72it/s]


19       CONTENTS\n CHAPTER I PAGE\n FORESTRY AND THE W...
158      A PAGE\n Abele Poplar for Town Planting, 101\n...
159      H\n Hard-wooded Trees for Economic Planting, 4...
160      T\n Tamarisk for Seaside Planting, 76-82\n _Ta...
430      TABLE 2[11]\nGROUP TAKING ALL TESTS AT ALL PER...
                               ...                        
18909    _Cæsar_, 125, 236.\n Camorra, the, 109.\n Capa...
18910    Darwin, 224.\n Daumer, Dr, 72.\n Decision, the...
18911    Habit, of prompt obedience, 20;\n in physical ...
18912    Padua, the Arena Chapel, 132.\n Parents, the e...
18913    Sadler, Professor, 221.\n Saviour, our, 145.\n...
Name: chunk, Length: 422, dtype: object

In [33]:
df = df[df["length"] <= 1000]
df

,chunk,length
0,THE CAVE OF ELEPHANTA.\n [Illustration: A view...,717
1,“The whole excavation consists of three princi...,699
2,But let us look a little further at the prophe...,679
3,“_Poiet_. I hope we shall have another good da...,645
4,“_Orn_. No such thing. The storm is their elem...,690
...,...,...
18904,LESSON.\n_Passage chosen_: LE CORBEAU.\n“Augus...,904
18905,MAP QUESTIONS.\nFrom the _Geographical Readers...,742
18906,LESSON.\n_Step 1._—Get the pupils to describe ...,763
18907,OBJECTS.\n1. To give the girls some idea of co...,738


In [34]:
cleaned_chunks = df.to_json(orient='records')

with open("/hpc/home/bfa6/work/github/yapper/dataset/cleaned_chunks.json", "w") as f:
    f.write(cleaned_chunks)


# Generating QA per chunk

In [9]:
# load chunks
with open("/hpc/home/bfa6/work/github/yapper/dataset/cleaned_chunks.json", "r") as f:
    cleaned_dataset = json.load(f) 

In [10]:
import random
random.Random(42).shuffle(cleaned_dataset)  # reproducible shuffle
len(cleaned_dataset)

18449

In [11]:
def dedupe_keep_first(rows, chunk_key="chunk"):
    seen = set()
    out = []
    for r in rows:
        c = r.get(chunk_key)
        if c is None:
            # if a row is missing the chunk key, keep it (or change behavior if you prefer)
            out.append(r)
            continue
        if c in seen:
            continue
        seen.add(c)
        out.append(r)
    return out

# Usage (after your shuffle)
cleaned_dataset = dedupe_keep_first(cleaned_dataset, chunk_key="chunk")
print("After dedupe, total unique rows:", len(cleaned_dataset))

After dedupe, total unique rows: 17383


In [12]:
n_train = 1000
n_eval  = 200
n_test  = 200

n_total = n_train + n_eval + n_test

cleaned_dataset = cleaned_dataset[:n_total]

train_dataset = cleaned_dataset[:n_train]
eval_dataset  = cleaned_dataset[n_train:n_train + n_eval]
test_dataset  = cleaned_dataset[n_train + n_eval:]

print(len(train_dataset), len(eval_dataset), len(test_dataset))
print(f"Total: {n_total}")

1000 200 200
Total: 1400


In [13]:
# --- 2. The Robust Processing Function ---

async def process_dataset_resumable(
    dataset: list, 
    output_path: str, 
    model: str = "gemini-2.5-flash-lite", 
    concurrency: int = 2, 
    delay: float = 5.0
):
    """
    Processes a dataset with resume capability.
    Saves results to a JSONL file immediately upon completion.
    """
    
    # A. Load existing progress to avoid re-doing work
    completed_texts = set()
    if os.path.exists(output_path):
        print(f"Found existing file at {output_path}. Checking progress...")
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    # We track progress using the chunk text itself as the key
                    completed_texts.add(data["chunk"])
                except json.JSONDecodeError:
                    continue
    
    # B. Filter the dataset
    # Only keep items where the chunk text is NOT in completed_texts
    # Note: This relies on 'chunk' text being unique. If duplicates exist in source, 
    # use a unique ID instead.
    tasks_to_do = [d for d in dataset if d["chunk"] not in completed_texts]
    
    print(f"Total items: {len(dataset)}")
    print(f"Already done: {len(completed_texts)}")
    print(f"Remaining: {len(tasks_to_do)}")
    
    if not tasks_to_do:
        print(f"All tasks for {output_path} are complete!")
        return

    # C. Setup Concurrency
    sem = asyncio.Semaphore(concurrency)
    file_lock = asyncio.Lock() # Prevents multiple tasks from writing to file at once

    async def process_and_save_chunk(chunk_data):
        async with sem:
            await asyncio.sleep(delay)
            
            # --- Your generation logic ---
            loop = asyncio.get_event_loop()
            try:
                qa = await loop.run_in_executor(
                    None,
                    generate_questions, # Assuming this function exists in your scope
                    chunk_data["chunk"],
                    model
                )
                
                
                result_entry = {
                    "chunk": chunk_data["chunk"],
                    "QAs": qa["QAs"],
                }

                if result_entry["QAs"] != []:
                    # --- D. Save Immediately ---
                    async with file_lock:
                        with open(output_path, "a", encoding="utf-8") as f:
                            f.write(json.dumps(result_entry) + "\n")
                        
                return result_entry
                
            except Exception as e:
                print(f"Error processing chunk: {e}")
                return None

    # Create tasks only for the remaining items
    tasks = [process_and_save_chunk(chunk) for chunk in tasks_to_do]
    
    # Run
    await tqdm_asyncio.gather(*tasks, desc=f"Processing {os.path.basename(output_path)}")


In [14]:
# --- 3. Execution ---

# Define your paths
base_path = "/hpc/home/bfa6/work/github/yapper/dataset"
train_path = f"{base_path}/trainset.jsonl"
eval_path = f"{base_path}/evalset.jsonl"
test_path = f"{base_path}/testset.jsonl"

# Run them sequentially (or gather them if you want distinct semaphores)
print("--- Starting Train Set ---")
await process_dataset_resumable(train_dataset, train_path)

print("--- Starting Eval Set ---")
await process_dataset_resumable(eval_dataset, eval_path)

print("--- Starting Test Set ---")
await process_dataset_resumable(test_dataset, test_path)

print("Done.")

--- Starting Train Set ---
Found existing file at /hpc/home/bfa6/work/github/yapper/dataset/trainset.jsonl. Checking progress...
Total items: 1000
Already done: 1000
Remaining: 0
All tasks for /hpc/home/bfa6/work/github/yapper/dataset/trainset.jsonl are complete!
--- Starting Eval Set ---
Found existing file at /hpc/home/bfa6/work/github/yapper/dataset/evalset.jsonl. Checking progress...
Total items: 200
Already done: 247
Remaining: 0
All tasks for /hpc/home/bfa6/work/github/yapper/dataset/evalset.jsonl are complete!
--- Starting Test Set ---
Found existing file at /hpc/home/bfa6/work/github/yapper/dataset/testset.jsonl. Checking progress...
Total items: 200
Already done: 259
Remaining: 0
All tasks for /hpc/home/bfa6/work/github/yapper/dataset/testset.jsonl are complete!
Done.


In [17]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows

def remove_overlaps(source_rows, target_rows, key="chunk"):
    source_chunks = {row.get(key) for row in source_rows}
    return [row for row in target_rows if row.get(key) not in source_chunks]


# --------- FILE PATHS ---------
train_path = "/hpc/home/bfa6/work/github/yapper/dataset/trainset.jsonl"
eval_path  = "/hpc/home/bfa6/work/github/yapper/dataset/evalset.jsonl"
test_path  = "/hpc/home/bfa6/work/github/yapper/dataset/testset.jsonl"

output_json = "/hpc/home/bfa6/work/github/yapper/dataset/splits.json"


# --------- LOAD DATA ---------
train_rows = load_jsonl(train_path)
eval_rows  = load_jsonl(eval_path)
test_rows  = load_jsonl(test_path)

print("Loaded:")
print(" train:", len(train_rows))
print(" eval :", len(eval_rows))
print(" test :", len(test_rows))


# --------- CLEAN OVERLAPS ---------
# Remove eval ∩ train
eval_clean = remove_overlaps(train_rows, eval_rows)

# Remove test ∩ train
test_clean = remove_overlaps(train_rows, test_rows)

# Remove test ∩ eval (after cleaning eval)
test_clean = remove_overlaps(eval_clean, test_clean)

def dedupe_keep_first(rows, chunk_key="chunk"):
    seen = set()
    out = []
    for r in rows:
        c = r.get(chunk_key)
        if c is None:
            # if a row is missing the chunk key, keep it (or change behavior if you prefer)
            out.append(r)
            continue
        if c in seen:
            continue
        seen.add(c)
        out.append(r)
    return out

# Usage (after your shuffle)
train_rows = dedupe_keep_first(train_rows, chunk_key="chunk")
print("After dedupe, total unique rows:", len(train_rows))

print("\nAfter cleaning:")
print(" train:", len(train_rows))
print(" eval :", len(eval_clean))
print(" test :", len(test_clean))

# --------- SAVE AS ONE JSON FILE ---------
splits = {
    "train": train_rows,
    "eval": eval_clean,
    "test": test_clean
}

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(splits, f, ensure_ascii=False, indent=2)

print(f"\nSaved cleaned splits to {output_json}")


Loaded:
 train: 1008
 eval : 252
 test : 261
After dedupe, total unique rows: 1000

After cleaning:
 train: 1000
 eval : 200
 test : 200

Saved cleaned splits to /hpc/home/bfa6/work/github/yapper/dataset/splits.json


In [21]:
import json

def check_five_qs_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for split in ["train", "eval", "test"]:
        rows = data.get(split, [])
        bad = [(i, len(r.get("QAs", []))) for i, r in enumerate(rows) if len(r.get("QAs", [])) != 5]

        print(f"\nChecking split: {split}")
        print(f"Total rows: {len(rows)}")

        if not bad:
            print("✔ All rows have exactly 5 QAs")
        else:
            print(f"✖ {len(bad)} rows do NOT have 5 QAs")
            print("Examples:")
            for idx, qcount in bad[:10]:
                print(f"  index {idx} → {qcount} QAs")


In [24]:
check_five_qs_json("/hpc/home/bfa6/work/github/yapper/dataset/splits.json")



Checking split: train
Total rows: 1000
✔ All rows have exactly 5 QAs

Checking split: eval
Total rows: 200
✔ All rows have exactly 5 QAs

Checking split: test
Total rows: 200
✔ All rows have exactly 5 QAs


In [23]:
import json

MODEL = "gemini-2.5-flash-lite"
SPLITS_PATH = "/hpc/home/bfa6/work/github/yapper/dataset/splits.json"

def regenerate_bad(rows):
    fixed = 0
    for r in rows:
        qas = r.get("QAs", [])
        if len(qas) != 5:
            print(f"Regenerating → current QAs: {len(qas)}")
            result = generate_questions(r["chunk"], MODEL)
            r["QAs"] = result.get("QAs", [])
            fixed += 1
    return fixed


# --- Load ---
with open(SPLITS_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# --- Fix ---
print("Checking TRAIN...")
fixed_train = regenerate_bad(data["train"])

print("Checking EVAL...")
fixed_eval = regenerate_bad(data["eval"])

print("Checking TEST...")
fixed_test = regenerate_bad(data["test"])

print("\nSummary:")
print("Train fixed:", fixed_train)
print("Eval fixed :", fixed_eval)
print("Test fixed :", fixed_test)

# --- Save back ---
with open(SPLITS_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("\nSaved cleaned splits.json ✔")


Checking TRAIN...
Regenerating → current QAs: 4
Checking EVAL...
Checking TEST...

Summary:
Train fixed: 1
Eval fixed : 0
Test fixed : 0

Saved cleaned splits.json ✔
